# Interactive all-pass ring resonator

This notebook accompanies the **Photonic Devices 2026** course.

It explores the spectral response of an **all-pass ring resonator**. The round-trip propagation factor is

$$
p(\lambda)=a\,e^{-j\phi(\lambda)},
$$

with

$$
\phi(\lambda)=\beta(\lambda)L,
\qquad
\beta(\lambda)=\frac{2\pi n_{\rm eff}(\lambda)}{\lambda}.
$$

The through-port transfer function is

$$
H_{\rm th}(\lambda)=\frac{t-p(\lambda)}{1-t\,p(\lambda)}.
$$

and the power transmission is

$$
T(\lambda)=|H_{\rm th}(\lambda)|^2.
$$

The notebook uses a first-order model for the wavelength dependence of the effective index. Around the reference wavelength $\lambda_0$,

$$
n_{\rm eff}(\lambda)
\simeq
n_{\rm eff,0}
+
\left.\frac{d n_{\rm eff}}{d\lambda}\right|_{\lambda_0}
(\lambda-\lambda_0).
$$

Using

$$
n_g=n_{\rm eff}-\lambda\frac{d n_{\rm eff}}{d\lambda}.
$$

the slope at $\lambda_0$ is

$$
\left.\frac{d n_{\rm eff}}{d\lambda}\right|_{\lambda_0}
=\frac{n_{\rm eff,0}-n_g}{\lambda_0}.
$$

The interactive controls allow you to change $t$, $L$, propagation loss, $n_{\rm eff,0}$, and $n_g$.

The reference wavelength is fixed at $\lambda_0=1.55\,\mu{\rm m}$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

plt.rcParams["figure.figsize"] = (8.2, 4.5)

c0 = 299792458.0       # speed of light [m/s]
lambda0_um = 1.55      # reference wavelength [um]


## 1. Physical model

The field self-coupling coefficient of the bus-ring coupler is $t$. For an ideal lossless coupler,

$$
t^2+\kappa^2=1.
$$

The propagation loss is specified as a **power attenuation coefficient in dB/cm**. If the ring round-trip length is $L$, the corresponding field attenuation after one round trip is

$$
a=10^{-\alpha_{\rm dB/cm}L_{\rm cm}/20}.
$$

At resonance,

$$
\phi=2\pi m,
$$

and therefore

$$
H_{\rm th,res}=\frac{t-a}{1-ta}.
$$

The three coupling regimes are:

- $t>a$: undercoupled;
- $t=a$: critical coupling;
- $t<a$: overcoupled.

At critical coupling, the through-port transmission ideally vanishes at resonance.

The free spectral range is approximately

$$
{\rm FSR}_f
\simeq
\frac{c}{n_gL},
$$

or, in wavelength,

$$
{\rm FSR}_\lambda
\simeq
\frac{\lambda_r^2}{n_gL}.
$$

For a sufficiently narrow resonance,

$$
\Delta\lambda_{\rm FWHM}
\simeq
{\rm FSR}_\lambda
\frac{1-ta}{\pi\sqrt{ta}},
$$

and

$$
Q=\frac{\lambda_r}{\Delta\lambda_{\rm FWHM}}.
$$


In [ ]:
def linear_neff(lambda_um, neff0, ng):
    """Linearized effective index around lambda0_um."""
    slope = (neff0 - ng) / lambda0_um
    return neff0 + slope * (lambda_um - lambda0_um)


def resonance_wavelength(m, L_um, neff0, ng):
    """Resonance wavelength [um] for the linearized-index model."""
    slope = (neff0 - ng) / lambda0_um
    denominator = m - slope * L_um
    if denominator <= 0:
        return np.nan
    return ng * L_um / denominator


def ring_quantities(t, L_um, loss_dB_cm, neff0, ng):
    """Return the main all-pass-ring quantities at the resonance nearest lambda0."""
    if not (0 < t < 1):
        raise ValueError("t must satisfy 0 < t < 1.")
    if L_um <= 0:
        raise ValueError("L must be positive.")
    if loss_dB_cm < 0:
        raise ValueError("Propagation loss cannot be negative.")
    if neff0 <= 0 or ng <= 0:
        raise ValueError("Refractive indices must be positive.")

    L_m = L_um * 1e-6
    L_cm = L_um * 1e-4

    a = 10.0 ** (-loss_dB_cm * L_cm / 20.0)
    kappa = np.sqrt(max(0.0, 1.0 - t**2))

    m0 = int(np.round(neff0 * L_um / lambda0_um))
    lambda_r_um = resonance_wavelength(m0, L_um, neff0, ng)

    FSR_f_Hz = c0 / (ng * L_m)
    FSR_lambda_um = lambda_r_um**2 / (ng * L_um)

    rt = t * a
    if rt <= 0:
        FWHM_um = np.nan
        finesse = np.nan
        Q = np.nan
    else:
        FWHM_um = (
            FSR_lambda_um
            * (1.0 - rt)
            / (np.pi * np.sqrt(rt))
        )
        finesse = FSR_lambda_um / FWHM_um
        Q = lambda_r_um / FWHM_um

    T_res = ((t - a) / (1.0 - t*a))**2

    if np.isclose(t, a, rtol=0.0, atol=2e-4):
        regime = "critical coupling"
    elif t > a:
        regime = "undercoupled"
    else:
        regime = "overcoupled"

    return {
        "a": a,
        "kappa": kappa,
        "m0": m0,
        "lambda_r_um": lambda_r_um,
        "FSR_f_Hz": FSR_f_Hz,
        "FSR_lambda_um": FSR_lambda_um,
        "FWHM_um": FWHM_um,
        "finesse": finesse,
        "Q": Q,
        "T_res": T_res,
        "regime": regime,
    }


def ring_spectrum(t, L_um, loss_dB_cm, neff0, ng, n_fsr=3.2, npts=12000):
    """Calculate power and phase response around lambda0."""
    q = ring_quantities(t, L_um, loss_dB_cm, neff0, ng)

    lambda_r_um = q["lambda_r_um"]
    FSR_lambda_um = q["FSR_lambda_um"]

    half_span = n_fsr * FSR_lambda_um
    lambda_min = max(0.05, lambda_r_um - half_span)
    lambda_max = lambda_r_um + half_span
    lambda_um = np.linspace(lambda_min, lambda_max, npts)

    neff = linear_neff(lambda_um, neff0, ng)
    beta_um = 2.0 * np.pi * neff / lambda_um
    phi = beta_um * L_um

    p = q["a"] * np.exp(-1j * phi)
    H = (t - p) / (1.0 - t*p)

    T = np.abs(H)**2
    phase = np.unwrap(np.angle(H))

    return lambda_um, T, phase, q


## 2. Interactive all-pass ring explorer

The wavelength window automatically follows the FSR, so several consecutive resonances remain visible as the ring parameters are changed.

The numerical summary reports the resonance nearest $\lambda_0$, the FSR, the estimated FWHM, finesse, $Q$, the round-trip attenuation $a$, and the coupling regime.


In [ ]:
def update_ring(t, L_um, loss_dB_cm, neff0, ng):
    try:
        lambda_um, T, phase, q = ring_spectrum(
            t=t,
            L_um=L_um,
            loss_dB_cm=loss_dB_cm,
            neff0=neff0,
            ng=ng,
        )
    except ValueError as exc:
        print(exc)
        return

    lambda_nm = 1e3 * lambda_um
    lambda_r_nm = 1e3 * q["lambda_r_um"]

    resonance_nm = []
    for m in range(q["m0"] - 8, q["m0"] + 9):
        lr = resonance_wavelength(m, L_um, neff0, ng)
        if np.isfinite(lr) and lambda_um[0] <= lr <= lambda_um[-1]:
            resonance_nm.append(1e3 * lr)

    print("ALL-PASS RING RESONATOR")
    print("-" * 52)
    print(f"Reference wavelength        = {1e3*lambda0_um:.1f} nm")
    print(f"Round-trip length L         = {L_um:.3f} um")
    print(f"Equivalent radius R         = {L_um/(2*np.pi):.3f} um")
    print(f"n_eff(lambda0)              = {neff0:.4f}")
    print(f"n_g                         = {ng:.4f}")
    print(f"Propagation loss            = {loss_dB_cm:.3f} dB/cm")
    print()
    print(f"Round-trip field factor a   = {q['a']:.6f}")
    print(f"Self-coupling t             = {t:.6f}")
    print(f"Cross-coupling kappa        = {q['kappa']:.6f}")
    print(f"Coupling regime             = {q['regime']}")
    print(f"Critical-coupling value     = t = a = {q['a']:.6f}")
    print()
    print(f"Nearest resonance           = {lambda_r_nm:.4f} nm")
    print(f"Resonance order m           = {q['m0']}")
    print(f"FSR                         = {q['FSR_f_Hz']/1e9:.3f} GHz")
    print(f"FSR                         = {1e3*q['FSR_lambda_um']:.4f} nm")
    print(f"FWHM                        = {1e3*q['FWHM_um']:.5f} nm")
    print(f"Finesse                     = {q['finesse']:.2f}")
    print(f"Q                           = {q['Q']:.3e}")
    print(f"Transmission at resonance   = {q['T_res']:.6f}")

    plt.figure(figsize=(8.6, 4.5))
    plt.plot(lambda_nm, T, linewidth=1.6)

    for lr_nm in resonance_nm:
        plt.axvline(lr_nm, linestyle=":", linewidth=0.8, alpha=0.55)

    plt.axvline(lambda_r_nm, linestyle="--", linewidth=1.1, label="selected resonance")
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Power transmission")
    plt.title(f"All-pass ring: {q['regime']}")
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8.6, 4.2))
    plt.plot(lambda_nm, phase, linewidth=1.6)
    plt.axvline(lambda_r_nm, linestyle="--", linewidth=1.1)
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Phase of H [rad]")
    plt.title("Through-port phase response")
    plt.grid(True, alpha=0.3)
    plt.show()


slider_style = {"description_width": "125px"}
slider_layout = widgets.Layout(width="390px")

t_slider = widgets.FloatSlider(
    value=0.9800,
    min=0.9000,
    max=0.9995,
    step=0.0005,
    description="t:",
    readout_format=".4f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

L_slider = widgets.FloatSlider(
    value=2*np.pi*10.0,
    min=20.0,
    max=300.0,
    step=0.2,
    description="L (um):",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

loss_slider = widgets.FloatSlider(
    value=27.93,
    min=0.0,
    max=100.0,
    step=0.05,
    description="loss (dB/cm):",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

neff_slider = widgets.FloatSlider(
    value=2.45,
    min=1.20,
    max=3.50,
    step=0.01,
    description="n_eff(lambda0):",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

ng_slider = widgets.FloatSlider(
    value=4.00,
    min=1.20,
    max=6.00,
    step=0.01,
    description="n_g:",
    readout_format=".2f",
    continuous_update=False,
    style=slider_style,
    layout=slider_layout,
)

controls = widgets.VBox([
    widgets.HBox([t_slider, L_slider]),
    widgets.HBox([loss_slider, neff_slider]),
    widgets.HBox([ng_slider]),
])

interactive_output = widgets.interactive_output(
    update_ring,
    {
        "t": t_slider,
        "L_um": L_slider,
        "loss_dB_cm": loss_slider,
        "neff0": neff_slider,
        "ng": ng_slider,
    },
)

display(controls, interactive_output)


## 3. Suggested exploration

Try to predict each trend before moving the sliders.

1. Increase $L$. Verify that the resonances become more closely spaced and that the FSR decreases.
2. Change $n_g$ while keeping the other parameters fixed. Verify the change in resonance spacing.
3. Change $n_{\rm eff,0}$. Observe that the absolute resonance positions shift while the FSR is mainly controlled by $n_g$.
4. Keep the propagation loss fixed and vary $t$ across the critical-coupling value $t=a$.
5. Compare the depth and phase response in the undercoupled, critically coupled, and overcoupled regimes.
6. Increase the propagation loss. Observe how the critical-coupling value changes and how the linewidth and $Q$ are affected.
7. Verify numerically that, close to critical coupling, the calculated FSR and FWHM agree with the analytical expressions used in the course.

This notebook is an analytical circuit-level model. It does not calculate the wavelength dependence of the coupler itself, bending radiation, back-reflection, or higher-order spatial modes.
